### Outputs: export the deliverables to the volume `outputs/` folder
`fact_trips.parquet/`, `dq_exclusions.csv`, `zone_stress.csv`, `zsi_windows.parquet/`

In [0]:
%run ../../config/config

In [0]:
%run ../../utils/zsi_functions

In [0]:
import os
from pyspark.sql.functions import col

os.makedirs(OUTPUT_DIR, exist_ok=True)

# fact_trips: one sorted part file; timestamps as TIMESTAMP_NTZ (wall clock, as given in the source)
spark.table(tbl("02_silver", "fact_trips")) \
     .withColumn("pickup_ts", col("pickup_ts").cast("timestamp_ntz")) \
     .withColumn("dropoff_ts", col("dropoff_ts").cast("timestamp_ntz")) \
     .orderBy("pickup_ts", "trip_key").coalesce(1) \
     .write.mode("overwrite").parquet(f"{OUTPUT_DIR}/fact_trips.parquet")

spark.table(tbl("03_gold", "zsi_windows")) \
     .orderBy("LocationID", "operational_date", "window_idx").coalesce(1) \
     .write.mode("overwrite").parquet(f"{OUTPUT_DIR}/zsi_windows.parquet")

# small tables -> single CSV files via pandas
to_pandas_safe(spark.sql(f"SELECT * FROM {tbl('02_silver', 'dq_exclusions')} ORDER BY rule_order")) \
    .to_csv(f"{OUTPUT_DIR}/dq_exclusions.csv", index=False)

to_pandas_safe(spark.sql(f"SELECT LocationID, zone, borough, S FROM {tbl('03_gold', 'zone_stress')} ORDER BY LocationID")) \
    .to_csv(f"{OUTPUT_DIR}/zone_stress.csv", index=False, float_format="%.10f")

In [0]:
display(dbutils.fs.ls(OUTPUT_DIR))